# Module 3: Sentiment & Execution Signal Extraction
One of the most powerful use cases for LLMs in execution is converting unstructured text (news, tweets) into structured data (JSON signals).


In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

load_dotenv()
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)


## 1. Define the Output Schema
We use Pydantic to strictly define the JSON output format we want the LLM to generate. This is crucial for integrating LLMs into automated Order Management Systems (OMS).


In [ ]:
class TradingSignal(BaseModel):
    asset: str = Field(description="The ticker symbol or asset name")
    sentiment: str = Field(description="Overall sentiment: POSITIVE, NEGATIVE, or NEUTRAL")
    signal: str = Field(description="Execution action: BUY, SELL, or HOLD")
    confidence: int = Field(description="Confidence score from 0 to 100")
    reasoning: str = Field(description="A 1-sentence explanation for the signal")

parser = JsonOutputParser(pydantic_object=TradingSignal)


## 2. Create the Extraction Prompt


In [ ]:
prompt = PromptTemplate(
    template="Analyze the following financial news and extract a execution signal.\n{format_instructions}\n\nNews: {news_text}\n",
    input_variables=["news_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser


## 3. Test on Mock News Data


In [ ]:
news_item = "CIPLA announces breakthrough approval for a new generic drug in the US market, expecting a 15% revenue bump this quarter."

result = chain.invoke({"news_text": news_item})
print("Extracted Signal:")
import json
print(json.dumps(result, indent=2))


Notice how the LLM perfectly structure the data, allowing your OMS bot to simply read `result['signal']`!
